# MaaS Policy & Rate Limit Testing

This notebook tests the core MaaS governance feature: **token-based rate limiting**.

We will:
1. Discover the MaaS endpoint
2. Create a subscription with a low token limit (e.g., 500 tokens/min)
3. Create an API key bound to that subscription
4. Verify inference works within the limit
5. Trigger rate limit exceeded (HTTP 429)
6. Clean up resources

**Prerequisites:**
- MaaS enabled on the cluster (`modelsAsService: Managed` in DataScienceCluster)
- PostgreSQL database configured (`maas-db-config` Secret)
- Model deployed and registered with MaaS (e.g., `qwen3-14b`)
- Cluster-admin or MaaS admin permissions

## 1. Setup — Discover MaaS Endpoint

In [1]:
import subprocess
import json
import time
import urllib.request
import ssl
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv("../.env")

# Discover cluster domain
CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    result = subprocess.run(
        ["oc", "get", "ingresses.config.openshift.io", "cluster",
         "-o", "jsonpath={.spec.domain}"],
        capture_output=True, text=True
    )
    CLUSTER_DOMAIN = result.stdout.strip()

MAAS_API = f"https://maas-api.{CLUSTER_DOMAIN}"

# Get OCP bearer token for admin operations
token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_result.stdout.strip()

# SSL context for self-signed certs
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print(f"MaaS API Endpoint: {MAAS_API}")
print(f"Cluster Domain:    {CLUSTER_DOMAIN}")
print(f"OCP User:          {subprocess.run(['oc', 'whoami'], capture_output=True, text=True).stdout.strip()}")

MaaS API Endpoint: https://maas-api.apps.openshift-cluster.sandbox1785.opentlc.com
Cluster Domain:    apps.openshift-cluster.sandbox1785.opentlc.com
OCP User:          kube:admin


In [2]:
def maas_request(path, method="GET", data=None, token=None):
    """Send request to MaaS API."""
    url = f"{MAAS_API}{path}"
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    else:
        headers["Authorization"] = f"Bearer {OC_TOKEN}"

    body = json.dumps(data).encode() if data else None
    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
            return resp.status, json.loads(resp.read().decode())
    except urllib.error.HTTPError as e:
        body = e.read().decode() if e.fp else ""
        try:
            return e.code, json.loads(body)
        except:
            return e.code, {"error": body}
    except Exception as e:
        return 0, {"error": str(e)}

# Test connectivity
status, resp = maas_request("/v1/models")
if status == 200:
    models = [m["id"] for m in resp.get("data", [])]
    print(f"\u2705 MaaS reachable. Available models: {models}")
else:
    print(f"\u26a0\ufe0f  MaaS returned {status}: {resp}")
    print("   Ensure MaaS is enabled and models are registered.")

⚠️  MaaS returned 500: {'error': ''}
   Ensure MaaS is enabled and models are registered.


## 2. Create a Subscription with Token Rate Limit

A MaaS subscription binds a rate limit policy to API keys. We create a test subscription with a very low limit (500 tokens/min) to easily trigger rate limiting.

In [ ]:
SUBSCRIPTION_NAME = "test-rate-limit-sub"
TOKEN_LIMIT = 500  # tokens per minute

# Create subscription with low token rate limit
sub_payload = {
    "name": SUBSCRIPTION_NAME,
    "limits": {
        "tokensPerMinute": TOKEN_LIMIT
    }
}

status, resp = maas_request("/maas-api/v1/subscriptions", method="POST", data=sub_payload)
if status in [200, 201]:
    SUBSCRIPTION_ID = resp.get("id", resp.get("name", SUBSCRIPTION_NAME))
    print(f"\u2705 Subscription created: {SUBSCRIPTION_ID}")
    print(f"   Token limit: {TOKEN_LIMIT} tokens/min")
    print(f"   Response: {json.dumps(resp, indent=2)}")
elif status == 409:
    print(f"\u26a0\ufe0f  Subscription '{SUBSCRIPTION_NAME}' already exists.")
    SUBSCRIPTION_ID = SUBSCRIPTION_NAME
else:
    print(f"\u274c Failed to create subscription ({status}): {resp}")
    SUBSCRIPTION_ID = None

## 3. Create an API Key Bound to Subscription

In [ ]:
# Create API key linked to the rate-limited subscription
key_payload = {
    "name": "test-rate-limit-key",
    "subscription": SUBSCRIPTION_ID,
    "expiresIn": "1h"
}

status, resp = maas_request("/maas-api/v1/api-keys", method="POST", data=key_payload)
if status in [200, 201]:
    API_KEY = resp.get("key", resp.get("token", ""))
    KEY_ID = resp.get("id", "")
    print(f"\u2705 API Key created (expires in 1h)")
    print(f"   Key ID: {KEY_ID}")
    print(f"   Key:    {API_KEY[:20]}...")
    print(f"   Bound to subscription: {SUBSCRIPTION_ID} ({TOKEN_LIMIT} tok/min)")
else:
    print(f"\u274c Failed to create API key ({status}): {resp}")
    API_KEY = None

## 4. Test Normal Request (Within Limit)

Send a small inference request that should succeed within the 500 tokens/min budget.

In [ ]:
MODEL_NS = os.getenv("MODEL_NAMESPACE", "rhoai-models")
MODEL_ID = "qwen3-14b"
INFERENCE_GW = f"https://inference-gateway.{CLUSTER_DOMAIN}/{MODEL_NS}/{MODEL_ID}"

def inference_request(path, data=None, token=None):
    """Send inference request via inference-gateway."""
    url = f"{INFERENCE_GW}{path}"
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"
    else:
        headers["Authorization"] = f"Bearer {OC_TOKEN}"
    body = json.dumps(data).encode() if data else None
    req = urllib.request.Request(url, data=body, headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
            return resp.status, json.loads(resp.read().decode())
    except urllib.error.HTTPError as e:
        body = e.read().decode() if e.fp else ""
        try:
            return e.code, json.loads(body)
        except:
            return e.code, {"error": body}
    except Exception as e:
        return 0, {"error": str(e)}

if not API_KEY:
    print("\u274c No API key available. Fix steps above first.")
else:
    chat_payload = {
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": "Say hello in one word."}],
        "max_tokens": 10
    }

    print(f"Inference via: {INFERENCE_GW}/v1/chat/completions")
    status, resp = inference_request("/v1/chat/completions",
                                     data=chat_payload, token=API_KEY)

    if status == 200:
        content = resp["choices"][0]["message"].get("content", "")
        usage = resp.get("usage", {})
        print(f"\u2705 Inference succeeded (within limit)")
        print(f"   Response: {content}")
        print(f"   Tokens used: {usage.get('total_tokens', 'N/A')}")
        print(f"   Budget: {TOKEN_LIMIT} tokens/min")
    elif status == 429:
        print(f"\u26a0\ufe0f  Rate limited on first request — token limit may be too low")
    else:
        print(f"\u274c Request failed ({status}): {resp}")

## 5. Test Rate Limit Exceeded (HTTP 429)

Send multiple requests to exhaust the token budget and trigger rate limiting.

In [ ]:
if not API_KEY:
    print("\u274c No API key available.")
else:
    print(f"Sending requests to exhaust {TOKEN_LIMIT} tokens/min budget...")
    print(f"Inference Gateway: {INFERENCE_GW}")
    print("=" * 60)

    rate_limited = False
    total_tokens_used = 0

    for i in range(20):
        chat_payload = {
            "model": MODEL_ID,
            "messages": [{"role": "user", "content": f"Count from 1 to 50. Request #{i+1}"}],
            "max_tokens": 100
        }

        status, resp = inference_request("/v1/chat/completions",
                                         data=chat_payload, token=API_KEY)

        if status == 200:
            usage = resp.get("usage", {})
            tokens = usage.get("total_tokens", 0)
            total_tokens_used += tokens
            print(f"  Request {i+1}: \u2705 OK (tokens: {tokens}, cumulative: {total_tokens_used})")
        elif status == 429:
            print(f"  Request {i+1}: \u26d4 HTTP 429 — Rate Limit Exceeded!")
            print(f"")
            print(f"\u2705 Rate limiting confirmed after {total_tokens_used} tokens")
            print(f"   Limit was: {TOKEN_LIMIT} tokens/min")
            if isinstance(resp, dict):
                print(f"   Response: {json.dumps(resp, indent=2)}")
            rate_limited = True
            break
        else:
            print(f"  Request {i+1}: \u274c Error {status}: {resp}")
            break

        time.sleep(0.5)

    if not rate_limited:
        print(f"\n\u26a0\ufe0f  Did not hit rate limit after {total_tokens_used} tokens.")
        print(f"   The limit ({TOKEN_LIMIT}/min) may not be enforced yet. Wait 1 min and retry.")

## 6. Cleanup

Delete the test API key and subscription.

In [ ]:
# Delete API key
if KEY_ID:
    status, resp = maas_request(f"/maas-api/v1/api-keys/{KEY_ID}", method="DELETE")
    if status in [200, 204]:
        print(f"\u2705 API key deleted: {KEY_ID}")
    else:
        print(f"\u26a0\ufe0f  Delete key returned {status}: {resp}")

# Delete subscription
if SUBSCRIPTION_ID:
    status, resp = maas_request(f"/maas-api/v1/subscriptions/{SUBSCRIPTION_ID}", method="DELETE")
    if status in [200, 204]:
        print(f"\u2705 Subscription deleted: {SUBSCRIPTION_ID}")
    else:
        print(f"\u26a0\ufe0f  Delete subscription returned {status}: {resp}")

print("\n\u2705 Cleanup complete.")

## Summary

| Test | Expected | Result |
|------|----------|--------|
| Create subscription (500 tok/min) | 201 Created | See Step 2 |
| Create API key | 201 Created | See Step 3 |
| Small request within limit | 200 OK | See Step 4 |
| Burst requests exceeding limit | 429 Too Many Requests | See Step 5 |
| Cleanup | 200/204 | See Step 6 |

This confirms that MaaS enforces token-based rate limiting per subscription, preventing any single API key from exhausting shared GPU resources.

## Next Steps

- Increase limits for production subscriptions in `3_maas_advanced.ipynb`
- Configure per-model access policies
- Monitor usage via Prometheus/Grafana dashboards